In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0" # whatever is free
# os.environ["JAX_PLATFORM_NAME"] = "cpu"
# os.environ["JAX_LOG_COMPILES"] = "1" # to see jit compiliation triggers

from loguru import logger
import jax
logger.info(f'JAX is using: {jax.devices()}')

2026-05-15 10:12:44.018 | INFO     | __main__:<module>:8 - JAX is using: [CudaDevice(id=0)]


In [ ]:
from pathlib import Path
from datetime import datetime
import wandb

from tqdm.auto import tqdm
import numpy as np
import jax.random as jr
import jax.numpy as jnp

from apa.utils.config import load_config
from apa.repo import RandomMatchingRepo

In [ ]:
config_folder = Path('../third_party/kinetix/configs')
assert config_folder.exists()
config = load_config(config_path=str(config_folder),config_name='naive_repo')


rng = jax.random.PRNGKey(config.get('seed',0))

In [6]:
USE_WANDB = config['use_wandb']

run_folder_name = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

if USE_WANDB:
    wandb_run = wandb.init(
        project="Naive Skill Repo", 
        config=dict(config)
    )

    wandb.define_metric("repo_step")
    # wandb.define_metric("pick_rate/*", step_metric="repo_step")
    wandb.define_metric("videos/*", step_metric="repo_step")
    wandb.define_metric("solve_rate/*", step_metric="repo_step")
    wandb.define_metric("ep_length/*", step_metric="repo_step")
    wandb.define_metric("ep_return/*", step_metric="repo_step")

    output_path = Path('outputs') / f'{run_folder_name}_{wandb_run.id}'
    
else:
    output_path = Path('outputs') / run_folder_name

output_path.mkdir(parents=True, exist_ok=False)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


wandb: Currently logged in as: sobhannili (sobhannili-epfl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# Env spawning

if config["train_level_mode"] == "random":
    rng, sub_rng = jr.split(rng)
    env_reset_states = get_random_envs(config["num_train_envs"], config["num_copies_per_env"], env, env_params, sub_rng)
    # unique_reset_states = jax.tree.map(lambda x: x[:, 0], reset_states)
else:
    env_reset_states = load_and_replicate_levels(
        config["train_levels_list"],
        int(config["num_copies_per_env"]),
    )
    # unique_reset_states = jax.tree.map(lambda x: x[:, 0], reset_states)

    config["num_train_envs"] = len(config["train_levels_list"])

Created static_env_params=StaticEnvParams(num_polygons=5, num_circles=2, num_joints=1, num_thrusters=1, max_polygon_vertices=4, num_solver_iterations=10, solver_batch_size=16, do_warm_starting=True, num_static_fixated_polys=4, screen_dim=(500, 500), downscale=4, frame_skip=2, max_shape_size=2, num_motor_bindings=4, num_thruster_bindings=2) when loading evaluation levels.


In [9]:
num_repo_updates = int(config['num_repo_updates'])
policy_repo = RandomMatchingRepo(config, env_reset_states, rng)
for repo_update_idx in tqdm(range(num_repo_updates)):
    policy_repo.advance()

  0%|          | 0/20 [00:00<?, ?it/s]

PPO Updates...:   0%|          | 0/100 [00:00<?, ?it/s]

PPO Updates...:   0%|          | 0/100 [00:00<?, ?it/s]

PPO Updates...:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x7f17cf7062f0>> (for post_run_cell), with arguments args (<ExecutionResult object at 7f17cf42f220, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7f17cf42ecb0, raw_cell="num_repo_updates = int(config['num_repo_updates']).." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://ssh-remote%2B7b22686f73744e616d65223a22414d476d30227d/mediaPFM/data/sobhan/SP2026_OpenEndedLearning/apa/components.ipynb#X13sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost

In [ ]:
# TODO:
# random actions baseline per env (for the same bumber of steps) -> per env get same stats
# option to load from the checkpoint path

In [ ]:
# import pickle

# with open(output_path / 'skill_repo_perf.pkl', 'wb') as file:
#     pickle.dump(repo_full_perf_history, file)

# # with open(output_path / 'skill_repo_perf.pkl', 'rb') as file:
# #     repo_full_perf_history = pickle.load(file)